# Gold analysis – business questions
Reads the Gold star schema only. Not part of the weekly job.

## Q1. When is a new company most at risk?
Purpose: the chance a new company is still registered after 1, 2 and 5 years, and the riskiest age.
Method (KPI spec v1): yearly deregistration probability per age band q = 1 − exp(−deregistrations ÷ company-years); survival S(N) = (1 − q₁) × … × (1 − q_N). Period method: assumes the last 12 months' conditions continue.
Tables: fct_exposure, dim_age_band, dim_company_classification, fct_company_snapshot, dim_date.
Caveats: deregistration ≠ default; deregistration dates are ASIC processing dates (batches, May–June 2026 spike).

In [0]:
-- Step 1: company-years at risk and deregistrations per age band (first 5 years)
WITH band_totals AS (
    SELECT
        a.age_band_key,
        a.age_band_label,
        k.borrower_segment,
        SUM(f.exposure_years)  AS exposure_years,
        SUM(f.deregistrations) AS deregistrations
    FROM workspace.sme_risk_gold.fct_exposure f
    JOIN workspace.sme_risk_gold.dim_age_band a
        ON f.age_band_key = a.age_band_key
    JOIN workspace.sme_risk_gold.dim_company_classification k
        ON f.classification_key = k.classification_key
    WHERE a.age_band_key BETWEEN 1 AND 5
    GROUP BY a.age_band_key, a.age_band_label, k.borrower_segment
),

-- Step 2: two groups – all companies, and trading companies only
groups AS (
    SELECT 'All companies' AS company_group, age_band_key, age_band_label,
           SUM(exposure_years) AS exposure_years, SUM(deregistrations) AS deregistrations
    FROM band_totals
    GROUP BY age_band_key, age_band_label
    UNION ALL
    SELECT 'Trading companies', age_band_key, age_band_label,
           SUM(exposure_years), SUM(deregistrations)
    FROM band_totals
    WHERE borrower_segment = 'Trading company'
    GROUP BY age_band_key, age_band_label
),

-- Step 3: rate (per company-year) and yearly probability
probabilities AS (
    SELECT
        company_group,
        age_band_key,
        age_band_label,
        deregistrations / exposure_years            AS dereg_rate,
        1 - EXP(-deregistrations / exposure_years)  AS yearly_dereg_prob
    FROM groups
)

-- Step 4: survival = running product of (1 - probability)
SELECT
    company_group,
    age_band_label,
    age_band_key                                                    AS survive_to_year,
    ROUND(dereg_rate * 100, 2)                                      AS dereg_rate_pct,
    ROUND(yearly_dereg_prob * 100, 2)                               AS yearly_dereg_prob_pct,
    ROUND(EXP(SUM(LN(1 - yearly_dereg_prob))
              OVER (PARTITION BY company_group ORDER BY age_band_key)) * 100, 1) AS survival_pct
FROM probabilities
ORDER BY company_group, age_band_key;

Databricks visualization. Run in Databricks to view.

In [0]:
-- Step 1: flag deregistrations processed in May-June 2026
WITH flagged AS (
    SELECT
        f.age_band_key,
        f.exposure_years,
        f.deregistrations,
        CASE
            WHEN f.deregistrations = 1 AND d.year_month IN ('2026-05', '2026-06') THEN 0
            ELSE f.deregistrations
        END AS deregistrations_excl_batch
    FROM workspace.sme_risk_gold.fct_exposure f
    JOIN workspace.sme_risk_gold.fct_company_snapshot s
        ON f.company_key = s.company_key
    JOIN workspace.sme_risk_gold.dim_date d
        ON s.deregistration_date_key = d.date_key
    WHERE f.age_band_key BETWEEN 1 AND 5
),

-- Step 2: totals per age band, with and without the batch
band_totals AS (
    SELECT
        age_band_key,
        SUM(exposure_years)              AS exposure_years,
        SUM(deregistrations)             AS deregistrations,
        SUM(deregistrations_excl_batch)  AS deregistrations_excl_batch
    FROM flagged
    GROUP BY age_band_key
),

-- Step 3: yearly probability, with and without the batch
probabilities AS (
    SELECT
        age_band_key,
        1 - EXP(-deregistrations            / exposure_years) AS prob_all,
        1 - EXP(-deregistrations_excl_batch / exposure_years) AS prob_excl_batch
    FROM band_totals
)

-- Step 4: survival, with and without the batch
SELECT
    age_band_key                                                              AS survive_to_year,
    ROUND(prob_all        * 100, 2)                                           AS yearly_dereg_prob_pct,
    ROUND(prob_excl_batch * 100, 2)                                           AS yearly_dereg_prob_excl_batch_pct,
    ROUND(EXP(SUM(LN(1 - prob_all))        OVER (ORDER BY age_band_key)) * 100, 1) AS survival_pct,
    ROUND(EXP(SUM(LN(1 - prob_excl_batch)) OVER (ORDER BY age_band_key)) * 100, 1) AS survival_excl_batch_pct
FROM probabilities
ORDER BY age_band_key;

In [0]:
-- Step 1: yearly probability per age band for each segment group
WITH band_probs AS (
    SELECT
        CASE WHEN k.borrower_segment = 'Trading company' THEN 'Trading company'
             ELSE 'All other segments' END                      AS segment_group,
        a.age_band_key,
        a.age_band_label,
        1 - EXP(-SUM(f.deregistrations) / SUM(f.exposure_years)) AS yearly_dereg_prob
    FROM workspace.sme_risk_gold.fct_exposure f
    JOIN workspace.sme_risk_gold.dim_age_band a
        ON f.age_band_key = a.age_band_key
    JOIN workspace.sme_risk_gold.dim_company_classification k
        ON f.classification_key = k.classification_key
    GROUP BY 1, a.age_band_key, a.age_band_label
),

-- Step 2: rank age bands within each group (1 = riskiest)
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY segment_group ORDER BY yearly_dereg_prob DESC) AS risk_rank
    FROM band_probs
)

-- Step 3: keep the riskiest band per group
SELECT
    segment_group,
    age_band_label                     AS riskiest_age_band,
    ROUND(yearly_dereg_prob * 100, 2)  AS yearly_dereg_prob_pct
FROM ranked
WHERE risk_rank = 1;